In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
import sklearn
import seaborn as sns

In [2]:
from datasets import load_dataset

REPO = "CMU-OSPO/protostar-climate-train-to-2018"
daily = load_dataset(REPO, "daily", split="train").to_pandas()
annual = load_dataset(REPO, "annual_indices", split="train").to_pandas()
cities = load_dataset(REPO, "city_info", split="train").to_pandas()

c:\Users\Saima\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
daily.head()

,ID,Name,Date,tmax,tmin,prcp
0,USW00014929,Aberdeen,1893-01-01,30.0,8.0,0.0
1,USW00014929,Aberdeen,1893-01-02,NaN,-10.0,0.0
2,USW00014929,Aberdeen,1893-01-03,24.0,-3.0,NaN
3,USW00014929,Aberdeen,1893-01-04,37.0,18.0,0.0
4,USW00014929,Aberdeen,1893-01-05,28.0,-10.0,0.0


In [4]:
annual.head()

,Name,ID,Lat,Lon,Year,Miss Tmax,Miss Tmin,Miss Prcp,Avg Temp,Avg Tmax,...,Max Tmin,Min Tmin,Warm Days,Warm Nights,Cold Days,Cold Nights,Max 1-day P,Max 5-day P,Wet Days,cdd
0,Aberdeen,USW00014929,45.4558,-98.4131,1893,18,17,34,41.09,55.14,...,72.0,-42.0,9.32,5.75,13.70,16.71,1.9,3.31,15.39,591.0
1,Aberdeen,USW00014929,45.4558,-98.4131,1894,127,127,64,49.05,62.58,...,75.0,-19.0,10.68,5.48,1.64,3.56,2.0,3.22,14.74,588.0
2,Aberdeen,USW00014929,45.4558,-98.4131,1895,11,9,0,41.94,56.58,...,70.0,-46.0,7.12,4.11,13.15,16.99,2.5,4.20,21.77,499.5
3,Aberdeen,USW00014929,45.4558,-98.4131,1896,7,7,0,41.16,53.84,...,68.0,-28.0,10.41,4.93,14.79,16.16,2.0,3.77,35.76,606.5
4,Aberdeen,USW00014929,45.4558,-98.4131,1897,11,15,2,40.89,53.71,...,76.0,-34.0,8.77,4.93,18.08,19.73,3.0,4.65,30.39,566.5


In [5]:
cities.head()

,Name,ID,Lat,Lon,Stn.Name,Stn.stDate,Stn.edDate
0,Lander,USW00024021,42.8153,-108.7261,LANDER WBO,1892-01-01,1946-05-28
1,Lander,USW00024021,42.8153,-108.7261,LANDER HUNT FIELD,1946-05-29,2018-12-31
2,Cheyenne,USW00024018,41.1519,-104.8061,CHEYENNE WBO,1871-01-01,1935-08-31
3,Cheyenne,USW00024018,41.1519,-104.8061,CHEYENNE MUNICIPAL ARPT,1935-09-01,2018-12-31
4,Wausau,USW00014897,44.9258,-89.6256,Wausau Record Herald,1896-01-01,1941-12-31


In [6]:
TARGETS = {
    "Avg Temp": ["Miss Tmax", "Miss Tmin"],
    "ToT Prcp": ["Miss Prcp"],
    "Max Tmax": ["Miss Tmax"],
    "cdd":      ["Miss Tmax", "Miss Tmin"],
}

df = annual.copy()
for t, miss in TARGETS.items():
    df.loc[(df[miss] > 10).any(axis=1), t] = np.nan

df = df[df["Year"] >= 1950].reset_index(drop=True)
print(df[list(TARGETS)].isna().mean())

def time_split(d, cutoff, horizon=7):
    train = d[d["Year"] <= cutoff]
    val   = d[(d["Year"] > cutoff) & (d["Year"] <= cutoff + horizon)]
    return train, val

folds = {"A": time_split(df, 2011), "B": time_split(df, 2004)}
for k, (tr, va) in folds.items():
    print(k, tr["Year"].agg(["min", "max"]).tolist(), va["Year"].agg(["min", "max"]).tolist(), len(tr), len(va))

Avg Temp    0.057557
ToT Prcp    0.051415
Max Tmax    0.052312
cdd         0.057557
dtype: float64
A [1950, 2011] [2012, 2018] 13020 1470
B [1950, 2004] [2005, 2011] 11550 1470


In [7]:
##Selecting numerical features
numerical_data1= daily.select_dtypes(include='number')
numerical_data2= cities.select_dtypes(include='number')
numerical_data3= annual.select_dtypes(include='number')
# Transposed stats for numerical features

numerical_data1.describe().T
numerical_data2.describe().T
numerical_data3.describe().T

c:\Users\Saima\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\Saima\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\Saima\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\Saima\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\Saima\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,count,mean,std,min,25%,50%,75%,max
Lat,28052.0,38.885431,5.318377,24.5550,34.8461,39.8328,42.8153,48.5614
Lon,28052.0,-93.736777,14.051975,-124.2367,-100.7603,-91.4442,-82.5550,-70.3003
Year,28052.0,1951.328640,39.237771,1851.0000,1918.0000,1952.0000,1985.0000,2018.0000
Miss Tmax,28052.0,20.757664,77.476750,0.0000,0.0000,0.0000,1.0000,366.0000
Miss Tmin,28052.0,20.725510,77.325212,0.0000,0.0000,0.0000,1.0000,366.0000
Miss Prcp,28052.0,18.662983,71.111263,0.0000,0.0000,0.0000,0.0000,366.0000
Avg Temp,26859.0,55.134448,9.072546,2.4300,48.4100,53.4200,62.2400,80.4200
Avg Tmax,26868.0,65.675203,9.215388,-0.5000,58.5200,64.1000,73.2100,93.8800
Avg Tmin,26878.0,44.585692,9.503268,-12.1000,37.9100,43.4400,51.2500,76.6400
ToT Prcp,28052.0,32.716196,17.189845,0.0000,19.3800,33.7200,43.9600,114.0300


In [10]:
# collect the categorical features of each dataset
datasets = {"cities": cities, "daily": daily, "annual": annual}
categorical_features = {}
for name, dataset in datasets.items():
    print(f"Categorical features in {name}:")
    categorical_data = dataset.select_dtypes(include=["object", "str"])
    categorical_features[name] = categorical_data.columns.tolist()
    if categorical_data.empty:
        print("  (none)")
    else:
        display(categorical_data.describe().T)

Categorical features in cities:


,count,unique,top,freq
Name,461,210,WestPalmBeach,10
ID,461,210,USW00012844,10
Stn.Name,461,430,WEST PALM BEACH,4
Stn.stDate,461,234,1893-01-01,38
Stn.edDate,461,199,2018-12-31,210


Categorical features in daily:


,count,unique,top,freq
ID,10245754,210,USW00023188,61361
Name,10245754,210,SanDiego,61361
Date,10245754,61361,1899-01-01,210


Categorical features in annual:


,count,unique,top,freq
Name,28052,210,SanDiego,168
ID,28052,210,USW00023188,168


In [11]:
cities.isna().sum()

Name          0
ID            0
Lat           0
Lon           0
Stn.Name      0
Stn.stDate    0
Stn.edDate    0
dtype: int64

In [ ]:
daily.isna().sum()

ID           0
Name         0
Date         0
tmax    582294
tmin    581392
prcp    523534
dtype: int64

In [14]:
annual.isna().sum()

Name              0
ID                0
Lat               0
Lon               0
Year              0
Miss Tmax         0
Miss Tmin         0
Miss Prcp         0
Avg Temp       1193
Avg Tmax       1184
Avg Tmin       1174
ToT Prcp          0
Max Tmax          0
Min Tmax          0
Max Tmin          0
Min Tmin          0
Warm Days         0
Warm Nights       0
Cold Days         0
Cold Nights       0
Max 1-day P       0
Max 5-day P       0
Wet Days          0
cdd            1206
dtype: int64